In [1]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
import gc

# Force a garbage collection sweep
gc.collect()
torch.cuda.empty_cache()
print(f"GPU Memory Wiped. Currently using: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU Memory Wiped. Currently using: 0.00 GB


In [2]:
%matplotlib inline
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
import sys
from abc import abstractmethod
from os.path import exists, join, basename
import time
import inspect
import math
import argparse
import enum
import yaml
import cv2
import torch
import torch.nn as nn
import torch.nn.init as init
import torchvision.utils as tvu
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from statsmodels.stats.proportion import proportion_confint
from tqdm import tqdm
from PIL import Image
import torchvision
from torchvision import transforms
import torchvision.datasets as datasets
from torchvision.io.image import decode_image
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.utils import draw_bounding_boxes
from torch.utils.data import DataLoader, Subset
from torchvision.transforms.functional import to_pil_image
from torch.distributions import Normal
import torchvision.ops.boxes as box_ops
from scipy.optimize import linear_sum_assignment
from scipy.stats import beta, norm
import numpy as np
from pycocotools.coco import COCO
assert torch.cuda.is_available(), "Error: CUDA is not available"
import random
device = "cuda" if torch.cuda.is_available() else "cpu"
# to ensure reproducibility
torch.manual_seed(0)
torch.cuda.manual_seed(0)


Mounted at /content/drive


In [3]:
if os.path.exists("/content/val2017.zip") == False:
  !sudo apt-get install -y axel
  #!axel -q http://images.cocodataset.org/zips/train2017.zip
  !axel -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
  !unzip -qq /content/train2017.zip -d /content/train2017/
  !unzip -qq /content/annotations_trainval2017.zip -d /content/annotations/
  !axel -q http://images.cocodataset.org/zips/val2017.zip
  !unzip -qq /content/val2017.zip -d /content/val2017/

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  axel
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 58.7 kB of archives.
After this operation, 204 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 axel amd64 2.17.11-1 [58.7 kB]
Fetched 58.7 kB in 1s (68.0 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package axel.
(Reading database ... 122354 files and directories currently inst

In [11]:
import os
import time
import argparse
from collections import OrderedDict
import json
import csv
import math

import torch
import torch.nn as nn
import torch.nn.init as init
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.transforms import functional as F
from PIL import Image

import scipy.stats as stats
from statsmodels.stats.proportion import proportion_confint
import pandas as pd

class DnCNN(nn.Module):
    def __init__(self, depth=17, n_channels=64, image_channels=3, use_bnorm=True, kernel_size=3):
        super(DnCNN, self).__init__()
        self.image_channels = image_channels
        padding = 1
        layers = []
        layers.append(nn.Conv2d(in_channels=image_channels, out_channels=n_channels, kernel_size=kernel_size, padding=padding, bias=True))
        layers.append(nn.ReLU(inplace=True))
        for _ in range(depth-2):
            layers.append(nn.Conv2d(in_channels=n_channels, out_channels=n_channels, kernel_size=kernel_size, padding=padding, bias=False))
            layers.append(nn.BatchNorm2d(n_channels, eps=0.0001, momentum = 0.95))
            layers.append(nn.ReLU(inplace=True))
        layers.append(nn.Conv2d(in_channels=n_channels, out_channels=image_channels, kernel_size=kernel_size, padding=padding, bias=False))
        self.dncnn = nn.Sequential(*layers)
        self._initialize_weights()

    def forward(self, x):
        y = x
        out = self.dncnn(x)
        return y - out

    def _initialize_weights(self):
        lastcnn = None
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                lastcnn = m
                init.orthogonal_(m.weight)
                if m.bias is not None:
                    init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                init.constant_(m.weight, 1)
                init.constant_(m.bias, 0)
        init.constant_(lastcnn.weight, 0)

def load_dncnn_denoiser(weights_path, device="cuda"):
    model = DnCNN(image_channels=3, depth=17, n_channels=64)
    checkpoint = torch.load(weights_path, map_location=device)
    state_dict = checkpoint['state_dict']
    clean_state_dict = OrderedDict()
    for k, v in state_dict.items():
        clean_key = k.replace('module.', '')
        clean_state_dict[clean_key] = v
    model.load_state_dict(clean_state_dict, strict=True)
    for param in model.parameters():
        param.requires_grad = False
    return model.to(device)

def ChiangGlobalBaselineDefense(image_tensor_01, model, dncnn_denoiser, device="cuda"):
    with torch.no_grad():
        noisy_batch = image_tensor_01.unsqueeze(0)
        purified_batch = dncnn_denoiser(noisy_batch)
        purified_batch = torch.clamp(purified_batch, 0.0, 1.0)
        outputs = model(purified_batch)

    B_global = []
    for box, score, label in zip(outputs[0]['boxes'], outputs[0]['scores'], outputs[0]['labels']):
        if score.item() >= 0.05: # Initial semantic filter
            B_global.append({
                'box': box.cpu().numpy().tolist(),
                'score': score.cpu().item(),
                'label': label.cpu().item()
            })
    return B_global

def compute_iou(box1, box2):
    """Calculates Intersection over Union for matching MC predictions."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter_area = max(0, x2 - x1) * max(0, y2 - y1)

    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    iou = inter_area / float(box1_area + box2_area - inter_area + 1e-6)
    return iou

def calculate_certified_radius(count, n, sigma, alpha=0.001):
    """Cohen/Chiang statistical bounds for Certified Radius."""
    if count == 0:
        return 0.0
    # Clopper-Pearson lower bound
    p_lower = proportion_confint(count, n, alpha=2*alpha, method="beta")[0]
    if p_lower > 0.5:
        return sigma * stats.norm.ppf(p_lower)
    return 0.0

def get_size_category(area):
    """Standard MS COCO size metrics."""
    if area < 32 * 32:
        return "Small"
    elif area <= 96 * 96:
        return "Medium"
    else:
        return "Large"

# =========================================================
# 3. MAIN EVALUATION LOOP
# =========================================================
def run_loop():
    parser = argparse.ArgumentParser()
    parser.add_argument("--img_dir", type=str, default="/content/val2017/val2017")
    parser.add_argument("--weights_path", type=str, default="/content/dncnn_twofive_clfstabobj.pth")
    parser.add_argument("--sigma", type=float, default=0.25)
    parser.add_argument("--N", type=int, default=100)
    parser.add_argument("--iou_thresh", type=float, default=0.50, help="Matching threshold for MC steps")
    args, unknown = parser.parse_known_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Running on Device: {device}")

    print("Loading Faster R-CNN...")
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    detector_model = fasterrcnn_resnet50_fpn_v2(weights=weights, box_score_thresh=0.05).to(device)
    detector_model.eval()

    print("Loading Chiang Baseline DnCNN Denoiser...")
    dncnn_denoiser = load_dncnn_denoiser(args.weights_path, device=device)
    dncnn_denoiser.eval()

    image_files = sorted([f for f in os.listdir(args.img_dir) if f.lower().endswith(('.jpg', '.png'))])

    # Open CSV for writing detailed object logs
    csv_file = open("chiang_detailed_objects.csv", mode='w', newline='')
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(["Image_Name", "Object_ID", "Label", "Area", "Size_Category", "MC_Count", "Certified_Radius"])
    img_count = 0
    for img_name in image_files:
        print(f"\nProcessing {img_name}...")
        img_path = os.path.join(args.img_dir, img_name)
        img = Image.open(img_path).convert("RGB")
        X_clean = F.to_tensor(img).to(device)

        # 1. Get the "Base" Ground Truth Predictions from the Clean Image
        # (This is standard practice: you must know what objects exist before you can certify them)
        base_predictions = ChiangGlobalBaselineDefense(X_clean, detector_model, dncnn_denoiser, device)

        # Initialize counts for each detected object
        mc_counts = [0] * len(base_predictions)

        start_time = time.time()

        # 2. The Randomized Smoothing Monte Carlo Loop
        for i in range(1, args.N + 1):
            noise = torch.randn_like(X_clean) * args.sigma
            X_noisy = torch.clamp(X_clean + noise, min=0.0, max=1.0)

            mc_boxes = ChiangGlobalBaselineDefense(X_noisy, detector_model, dncnn_denoiser, device)

            # Match MC boxes to Base boxes
            for base_idx, base_obj in enumerate(base_predictions):
                for mc_obj in mc_boxes:
                    if base_obj['label'] == mc_obj['label']:
                        if compute_iou(base_obj['box'], mc_obj['box']) >= args.iou_thresh:
                            mc_counts[base_idx] += 1
                            break # Found a match, move to next base object

        elapsed = time.time() - start_time
        print(f"Finished 100 MC steps for image {img_count} {img_name} in {elapsed:.2f} seconds. Certifying {len(base_predictions)} objects.")

        for idx, base_obj in enumerate(base_predictions):
            box = base_obj['box']
            area = (box[2] - box[0]) * (box[3] - box[1])
            size_cat = get_size_category(area)
            count = mc_counts[idx]

            radius = calculate_certified_radius(count, args.N, args.sigma)

            csv_writer.writerow([img_name, idx, base_obj['label'], area, size_cat, count, radius])
            csv_file.flush() # Force write to Drive so you don't lose data on crash
        img_count = img_count + 1
        if img_count == 500:
          break
    csv_file.close()

    print("\n--- Generating Final ACR Summary ---")
    df = pd.read_csv("chiang_detailed_objects.csv")
    summary = df.groupby('Size_Category')['Certified_Radius'].mean().reset_index()
    summary.rename(columns={'Certified_Radius': 'ACR'}, inplace=True)
    summary.to_csv("chiang_acr_summary.csv", index=False)
    print(summary.to_string(index=False))
    print("\nSUCCESS! Saved to chiang_detailed_objects.csv and chiang_acr_summary.csv")




In [12]:
run_loop()

Running on Device: cuda
Loading Faster R-CNN...
Loading Chiang Baseline DnCNN Denoiser...

Processing 000000000139.jpg...
Finished 100 MC steps for image 0 000000000139.jpg in 2.55 seconds. Certifying 66 objects.

Processing 000000000285.jpg...
Finished 100 MC steps for image 1 000000000285.jpg in 2.22 seconds. Certifying 5 objects.

Processing 000000000632.jpg...
Finished 100 MC steps for image 2 000000000632.jpg in 2.91 seconds. Certifying 100 objects.

Processing 000000000724.jpg...
Finished 100 MC steps for image 3 000000000724.jpg in 1.96 seconds. Certifying 25 objects.

Processing 000000000776.jpg...
Finished 100 MC steps for image 4 000000000776.jpg in 2.18 seconds. Certifying 26 objects.

Processing 000000000785.jpg...
Finished 100 MC steps for image 5 000000000785.jpg in 2.16 seconds. Certifying 6 objects.

Processing 000000000802.jpg...
Finished 100 MC steps for image 6 000000000802.jpg in 2.16 seconds. Certifying 7 objects.

Processing 000000000872.jpg...
Finished 100 MC ste

In [13]:
df = pd.read_csv("chiang_detailed_objects.csv")
summary = df.groupby('Size_Category')['Certified_Radius'].mean().reset_index()
summary.rename(columns={'Certified_Radius': 'ACR'}, inplace=True)
summary.to_csv("chiang_acr_summary.csv", index=False)
print(summary.to_string(index=False))
print("\nSUCCESS! Saved to chiang_detailed_objects.csv and chiang_acr_summary.csv")

Size_Category      ACR
        Large 0.144411
       Medium 0.108047
        Small 0.086844

SUCCESS! Saved to chiang_detailed_objects.csv and chiang_acr_summary.csv
